In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/the-ancient-texts-provenance-challenge/sample_submission.csv
/kaggle/input/the-ancient-texts-provenance-challenge/train.csv
/kaggle/input/the-ancient-texts-provenance-challenge/test.csv


# Importing Modules

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer,TfidfTransformer

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import MinMaxScaler,MaxAbsScaler,StandardScaler,FunctionTransformer

# Loading Dataset

In [4]:
df=pd.read_csv("/kaggle/input/the-ancient-texts-provenance-challenge/train.csv")

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline,Pipeline

In [6]:
X=pd.DataFrame(df[['text']])
y=df['label']

In [7]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
ct=ColumnTransformer([
    ('tfidf', TfidfVectorizer(max_features=1000), ['text']),
])

In [9]:
vectorizer = CountVectorizer(stop_words='english')
trainVectorizerArray =   vectorizer.fit_transform(X_train['text'])

transformer = TfidfTransformer()
res = transformer.fit_transform(trainVectorizerArray)


In [10]:
model=LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(res,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [11]:
from sklearn.metrics import f1_score

In [12]:
tdf=pd.read_csv("/kaggle/input/the-ancient-texts-provenance-challenge/test.csv")

In [13]:
X_test_vect=vectorizer.transform(X_test['text'])
X_test_res=transformer.transform(X_test_vect)

In [14]:
test_vect=vectorizer.transform(tdf['text'])
test_res=transformer.transform(test_vect)

In [15]:
f1_score(y_train,model.predict(res),average='macro')

0.7026662101741669

In [16]:
f1_score(y_test,model.predict(X_test_res),average='macro')

0.43735051889559956

In [17]:
pred=model.predict(test_res)

In [18]:
y_pred=pd.DataFrame(pred,columns=['label'])
sub=pd.concat([tdf['id'],y_pred],axis=1)
sub.to_csv("submission.csv",index=False)